In [ ]:
import logging
import os
from datetime import datetime

import numpy as np
import pandas as pd
from src.functions import RenamerExifTool, check_column_consistency
from tqdm.auto import tqdm

In [ ]:
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(message)s"
)
# pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Parameters

In [ ]:
# Directories
DIR_SC = "sc"  # Source
DIR_ST = "st"  # Staging
DIR_FT = "ft"  # Fact

# Mapping of filenames before to after to allow for reversibility
current_time = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
PATH_CSV = f"mapping_{current_time}.csv"

# Init

In [ ]:
renamer = RenamerExifTool(dir_sc=DIR_SC)

# Create df without actual renaming

In [ ]:
df = renamer.run(dir_st=DIR_ST, dir_ft=DIR_FT, path_csv=PATH_CSV, rename=False)
print(df.shape)

In [ ]:
df.head(2)

In [ ]:
df.tail(2)

# Count warnings

In [ ]:
df["warning_msg"] = df["warning_msg"].fillna("")
if df[df["warning_msg"] != ""].shape[0] == 0:
    print("No warnings")
if df[df["warning_msg"] != ""].shape[0] > 0:
    df[df["warning_msg"] != ""][cols].head()

# Unique info

In [ ]:
quicktime = df.get("QuickTime:Model", pd.Series(np.nan, index=df.index))
exif = df.get("EXIF:Model", pd.Series(np.nan, index=df.index))

df["manufacturer"] = np.where(
    quicktime.notnull() & (quicktime != ""),
    quicktime,
    np.where(exif.notnull() & (exif != ""), exif, "Unknown"),
)

df.groupby("manufacturer")["info_msg"].value_counts(dropna=False)

# Check consistency of files in directories

## Photo

In [ ]:
df_photo = df[
    df["File:FileTypeExtension"].str.lower().isin(["arw", "dng", "nef", "x3f"])
]


columns_to_check = [
    "File:FileTypeExtension",
    "EXIF:Make",
    "EXIF:Model",
    "EXIF:ImageWidth",
    "EXIF:ImageHeight",
]
check_column_consistency(
    df=df_photo, columns_to_check=columns_to_check, column_pairs=None
)

## Video

In [ ]:
df_video = df[df["File:FileTypeExtension"].str.lower().isin(["avi", "mov", "mp4"])]


columns_to_check = [
    "File:FileTypeExtension",
    "QuickTime:ImageWidth",
    "QuickTime:ImageHeight",
    "QuickTime:VideoFrameRate",
    "QuickTime:Model",
    "QuickTime:Sharpness",
    "QuickTime:ColorMode",
    "QuickTime:MaximumShutterAngle",
    "QuickTime:AutoISOMax",
    "QuickTime:AutoISOMin",
    "QuickTime:FieldOfView",
    "QuickTime:GoPro_EISE",
    "QuickTime:GoPro_LOGB",
    "QuickTime:GoPro_NR01",
    "QuickTime:GoPro_EXPX",
    "QuickTime:GoPro_EXPN",
    "QuickTime:GoPro_EXPQ",
]
column_pairs = [("QuickTime:AutoISOMax", "QuickTime:AutoISOMin")]
check_column_consistency(df_video, columns_to_check)

# Number of files to rename

In [ ]:
n_rename = df[df["File:FileName"] != df["filename_new"]].shape[0]
if n_rename == 0:
    print("No files to rename")
elif n_rename > 0:
    print(f"{n_rename} files to rename")

# Rename

In [ ]:
assert df[df["warning_msg"] != ""].shape[0] == 0
renamer.run(output=True)